In [8]:
# ============================================
# 깨끗한 재시작: 5-Seed × 5-Fold 앙상블
# ============================================
import pandas as pd
import numpy as np
import lightgbm
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

In [9]:
# ============================================
# 1. 데이터 로드
# ============================================
train_raw = pd.read_csv('./data/train.csv')
test_raw  = pd.read_csv('./data/test.csv')

print(f"train: {train_raw.shape}, test: {test_raw.shape}")
print(f"target 분포: {train_raw[TARGET].value_counts(normalize=True).round(3).to_dict()}")

# test에 파생변수용 컬럼 있는지 확인
for col in ['총 출산 횟수', '해동된 배아 수']:
    exists = col in test_raw.columns
    print(f"  test에 '{col}' 존재?: {exists}")

train: (256351, 69), test: (90067, 68)
target 분포: {0: 0.742, 1: 0.258}
  test에 '총 출산 횟수' 존재?: True
  test에 '해동된 배아 수' 존재?: True


In [10]:
# ============================================
# 2. 최소 전처리 (NaN 보존 원칙)
# ============================================
def prepare_data(df):
    """NaN을 절대 fillna하지 않는 최소 전처리"""
    df = df.copy()

    # 파생변수 1: ever_delivered (문헌 최강 긍정 신호)
    if '총 출산 횟수' in df.columns:
        births = df['총 출산 횟수'].astype(str).str.extract(r'(\d+)')[0]
        births_num = pd.to_numeric(births, errors='coerce').fillna(0)
        df['ever_delivered'] = (births_num > 0).astype(int)

    # 파생변수 2: is_FET (주기 구분)
    if '해동된 배아 수' in df.columns:
        df['is_FET'] = (df['해동된 배아 수'].fillna(0) > 0).astype(int)

    # 횟수 컬럼 문자열 -> 숫자 (NaN 그대로 유지)
    count_cols = [
        '총 시술 횟수', '클리닉 내 총 시술 횟수',
        'IVF 시술 횟수', 'DI 시술 횟수',
        '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
        '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수',
    ]
    for col in count_cols:
        if col in df.columns:
            extracted = df[col].astype(str).str.extract(r'(\d+)')[0]
            df[col] = pd.to_numeric(extracted, errors='coerce')  # NaN 보존

    return df

train = prepare_data(train_raw)
test  = prepare_data(test_raw)

print(f"\n전처리 후 shape: train {train.shape}, test {test.shape}")



전처리 후 shape: train (256351, 71), test (90067, 70)


In [11]:
# ============================================
# 3. 학습 데이터 구성 (object -> category)
# ============================================
X = train.drop(columns=[ID_COL, TARGET]).copy()
y = train[TARGET].copy()
X_test = test.drop(columns=[ID_COL]).copy()

# object 컬럼을 category로 변환 (LightGBM 네이티브 NaN 처리)
obj_cols = X.select_dtypes(include='object').columns.tolist()
print(f"\ncategory 변환 대상 ({len(obj_cols)}개): {obj_cols[:5]}...")

for col in obj_cols:
    X[col] = X[col].astype('category')
    if col in X_test.columns:
        X_test[col] = X_test[col].astype('category')

# train과 test의 category 레벨 일치시키기
for col in obj_cols:
    if col not in X_test.columns:
        continue
    tr_cats = X[col].cat.categories.tolist()
    te_cats = X_test[col].cat.categories.tolist()
    all_cats = sorted(set(tr_cats) | set(te_cats), key=lambda x: (x is None, str(x)))
    X[col] = X[col].cat.set_categories(all_cats)
    X_test[col] = X_test[col].cat.set_categories(all_cats)

# NaN 여전히 살아있는지 확인 (중요!)
nan_check = X.isnull().sum().sort_values(ascending=False).head(5)
print(f"\nNaN 살아있는 상위 5개 컬럼 (있어야 정상):")
print(nan_check)

print(f"\n최종: X {X.shape}, y {y.shape}, X_test {X_test.shape}")



category 변환 대상 (10개): ['시술 시기 코드', '시술 당시 나이', '시술 유형', '특정 시술 유형', '배란 유도 유형']...

NaN 살아있는 상위 5개 컬럼 (있어야 정상):
난자 해동 경과일                254915
PGS 시술 여부                254422
PGD 시술 여부                254172
착상 전 유전 검사 사용 여부         253633
임신 시도 또는 마지막 임신 경과 연수    246981
dtype: int64

최종: X (256351, 69), y (256351,), X_test (90067, 69)


In [12]:
# ============================================
# 4. 5-Seed × 5-Fold 앙상블
# ============================================
SEEDS   = [42, 2024, 777, 1234, 31337]
N_FOLDS = 5

# 명확한 변수명 사용 (덮어쓰기 방지)
final_oof_preds  = np.zeros(len(X))
final_test_preds = np.zeros(len(X_test))
seed_aucs = []

print(f"\n{'='*60}")
print(f"5-Seed × {N_FOLDS}-Fold LightGBM 앙상블 시작")
print(f"{'='*60}")

for seed_idx, seed in enumerate(SEEDS):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    this_seed_oof  = np.zeros(len(X))
    this_seed_test = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        model = LGBMClassifier(
            n_estimators      = 1000,
            learning_rate     = 0.05,
            num_leaves        = 63,
            min_child_samples = 20,
            subsample         = 0.8,
            colsample_bytree  = 0.8,
            reg_alpha         = 0.1,
            reg_lambda        = 0.1,
            random_state      = seed,
            verbose           = -1,
            n_jobs            = -1,
        )
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
            callbacks=[
                lightgbm.early_stopping(50, verbose=False),
                lightgbm.log_evaluation(0),
            ]
        )
        this_seed_oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
        this_seed_test       += model.predict_proba(X_test)[:, 1] / N_FOLDS

    seed_auc = roc_auc_score(y, this_seed_oof)
    seed_aucs.append(seed_auc)
    print(f"Seed {seed:>5}: OOF AUC = {seed_auc:.5f}")

    # 최종 앙상블에 누적
    final_oof_preds  += this_seed_oof  / len(SEEDS)
    final_test_preds += this_seed_test / len(SEEDS)



5-Seed × 5-Fold LightGBM 앙상블 시작
Seed    42: OOF AUC = 0.73939
Seed  2024: OOF AUC = 0.73973
Seed   777: OOF AUC = 0.73990
Seed  1234: OOF AUC = 0.73975
Seed 31337: OOF AUC = 0.73929


In [13]:
# ============================================
# 5. 결과 출력 & 저장
# ============================================
final_auc = roc_auc_score(y, final_oof_preds)

print(f"\n{'='*60}")
print(f"Seed별 OOF AUC 평균: {np.mean(seed_aucs):.5f}")
print(f"최종 앙상블 OOF AUC: {final_auc:.5f}")
print(f"앙상블 이득:         +{final_auc - np.mean(seed_aucs):.5f}")
print(f"{'='*60}")

# 제출 파일 생성
sub = pd.read_csv('./data/sample_submission.csv')
sub['probability'] = final_test_preds

filename = f"submission_{datetime.now().strftime('%Y%m%d_%H%M')}_v4.csv"
sub.to_csv(filename, index=False)

print(f"\n저장: {filename}")
print(f"확률 min/mean/max: {final_test_preds.min():.4f} / {final_test_preds.mean():.4f} / {final_test_preds.max():.4f}")
print(f"확률 고유값 개수: {len(np.unique(np.round(final_test_preds, 8)))}/{len(final_test_preds)}")

# OOF 저장 (나중에 앙상블용)
np.save(f'oof_v4_auc_{final_auc:.5f}.npy', final_oof_preds)
np.save(f'test_v4_auc_{final_auc:.5f}.npy', final_test_preds)
print(f"OOF 저장: oof_v4_auc_{final_auc:.5f}.npy")


Seed별 OOF AUC 평균: 0.73961
최종 앙상블 OOF AUC: 0.74033
앙상블 이득:         +0.00072

저장: submission_20260424_2200_v4.csv
확률 min/mean/max: 0.0008 / 0.2584 / 0.6969
확률 고유값 개수: 88961/90067
OOF 저장: oof_v4_auc_0.74033.npy
